In [4]:
import h5py
path = "./dataset/results_batch_00_sampling_0.hdf5"
with h5py.File(path, "r") as f:
    def show(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"[DATASET] {name} shape={obj.shape} dtype={obj.dtype}")
        else:
            print(f"[GROUP  ] {name}")
    f.visititems(show)

import h5py
import numpy as np

KEYS = [
    "APs_complete","LANs_complete","angMomentum_complete","aratios_complete",
    "eccs_complete","energy_complete","incs_complete","masses_complete",
    "mratios_complete","periods_complete","positions_complete",
    "smaxes_complete","tAnos_complete","times_complete","velocities_complete"
]

def load_one_h5(path, T=None, dtype=np.float32):
    sl = slice(None) if T is None else slice(0, T)
    out = {}
    with h5py.File(path, "r") as f:
        for k in KEYS:
            if k == "times_complete":
                out[k] = f[k][:, sl].astype(dtype)              # (N,T)
            else:
                # 其它都是 (N,T,...) 或 (N,T,3,3)
                out[k] = f[k][:, sl, ...].astype(dtype)
    return out

paths = [
    "./dataset/results_batch_00_sampling_0.hdf5",
    "./dataset/results_batch_01_sampling_0.hdf5",
    "./dataset/results_batch_02_sampling_0.hdf5",
    "./dataset/results_batch_03_sampling_0.hdf5",
    "./dataset/results_batch_04_sampling_0.hdf5",
]

# optional: limit to first T steps for faster loading and prototyping
T = 200

dicts = [load_one_h5(p, T=T) for p in paths]

data = {k: np.concatenate([d[k] for d in dicts], axis=0) for k in KEYS}

print(data["positions_complete"].shape)  # (N_total, T, 3, 3)
print("data load complete")



[DATASET] APs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] LANs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] angMomentum_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] aratios_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] eccs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] energy_complete shape=(8000, 1001, 1) dtype=float64
[DATASET] incs_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] masses_complete shape=(8000, 1001, 3) dtype=float64
[DATASET] mratios_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] periods_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] positions_complete shape=(8000, 1001, 3, 3) dtype=float64
[DATASET] smaxes_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] tAnos_complete shape=(8000, 1001, 2) dtype=float64
[DATASET] times_complete shape=(8000, 1001) dtype=float64
[DATASET] velocities_complete shape=(8000, 1001, 3, 3) dtype=float64
(40000, 200, 3, 3)
data load complete


In [5]:
# check if we get the same data as before
N = 8000*5  # 5 files, each with 5000 systems
assert data["positions_complete"].shape == (N, T, 3, 3)
assert data["velocities_complete"].shape == (N, T, 3, 3)
assert data["times_complete"].shape == (N, T)


In [6]:
import pandas as pd
import torch

df = pd.read_csv("dataset/completed_systems_sampling_0.txt", sep=r"\s+", header=None,
                 names=["index","num_batch","time_unbound","ignore_other_than-1","dignosis"])

# datatype
df = df.astype({"index":"int64","num_batch":"int64","time_unbound":"float32","ignore_other_than-1":"float32","dignosis":"int64"})

arr = df.to_numpy()
print(df.columns)
print(arr.shape, arr.dtype)
print(arr[:10, :5])  
#tensor = torch.from_numpy(arr)  
#print(tensor.shape, tensor.dtype)


Index(['index', 'num_batch', 'time_unbound', 'ignore_other_than-1',
       'dignosis'],
      dtype='str')
(40000, 5) float64
[[360.           0.          10.           1.2236681    0.        ]
 [179.           0.           5.           1.22473073   0.        ]
 [183.           0.           5.           1.24002266   0.        ]
 [361.           0.          20.           1.27451062   0.        ]
 [178.           0.          30.           1.24214554   0.        ]
 [349.           0.          15.           1.26477361   0.        ]
 [180.           0.          20.           1.25246263   0.        ]
 [176.           0.          20.           1.27163601   0.        ]
 [367.           0.          65.           1.30586576   0.        ]
 [182.           0.          30.           1.32230663   0.        ]]


In [7]:
def nan_report(data, T=200):
    rep = []
    N = next(iter(data.values())).shape[0]
    for k, a in data.items():
        b = a[:, :T, ...] if (a.ndim >= 2 and a.shape[1] >= T) else a
        bad = ~np.isfinite(b).all(axis=tuple(range(1, b.ndim)))
        rep.append((k, int(bad.sum()), N))
    return sorted(rep, key=lambda x: x[1], reverse=True)

for k, bad, N in nan_report(data, T=100)[:]:
    print(k, bad, "/", N)


periods_complete 16847 / 40000
tAnos_complete 14509 / 40000
positions_complete 254 / 40000
APs_complete 0 / 40000
LANs_complete 0 / 40000
angMomentum_complete 0 / 40000
aratios_complete 0 / 40000
eccs_complete 0 / 40000
energy_complete 0 / 40000
incs_complete 0 / 40000
masses_complete 0 / 40000
mratios_complete 0 / 40000
smaxes_complete 0 / 40000
times_complete 0 / 40000
velocities_complete 0 / 40000


## 1D CNN

In [8]:
"""
End-to-end: build (B, C, T) features from your dataset + train a 1D-CNN classifier
using the first T=100 steps.

Features included (per time step):
A) Precomputed orbital/physical series (with angle sin/cos):
  - aratios (1)
  - mratios (2)
  - masses (3)
  - eccs (2)
  - smaxes (2)
  - periods (2)
  - energy (1)
  - angMomentum (3)
  - APs (2) -> sin/cos (4)
  - LANs (2) -> sin/cos (4)
  - tAnos (2) -> sin/cos (4)
  - incs (2) -> sin/cos (4)

B) Geometry from positions/velocities (translation/rotation invariant):
  - pairwise distances: r12, r13, r23 (3)
  - pairwise relative speed magnitudes: |v12|, |v13|, |v23| (3)
  - pairwise radial speeds: rdot12, rdot13, rdot23 (3)

C) Optional: normalized time channel (1)

Total channels ~ 32 (A) + 9 (B) + 1 (time) = ~42 channels.

You provide:
  - data: dict of numpy arrays with the keys shown in your printout
  - y: numpy array (N,) labels 0/1
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# ---------------------------
# Feature engineering helpers
# ---------------------------

def sincos(x: np.ndarray) -> np.ndarray:
    """x: (..., k) angles in radians. Return (..., 2k) with [sin, cos]."""
    x = x.astype(np.float32)
    return np.concatenate([np.sin(x), np.cos(x)], axis=-1)

def pairwise_geom(pos: np.ndarray, vel: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    pos: (N, T, 3, 3)  -> bodies, xyz
    vel: (N, T, 3, 3)
    Return geom features per step: (N, T, 9)
      [r12,r13,r23, v12,v13,v23, rdot12,rdot13,rdot23]
    All are coordinate-frame invariant (translation cancels in differences; rotation cancels in norms/dots).
    """
    # differences: dij = xi - xj
    x1, x2, x3 = pos[:, :, 0, :], pos[:, :, 1, :], pos[:, :, 2, :]
    v1, v2, v3 = vel[:, :, 0, :], vel[:, :, 1, :], vel[:, :, 2, :]

    d12 = x1 - x2
    d13 = x1 - x3
    d23 = x2 - x3

    u12 = v1 - v2
    u13 = v1 - v3
    u23 = v2 - v3

    r12 = np.linalg.norm(d12, axis=-1)  # (N,T)
    r13 = np.linalg.norm(d13, axis=-1)
    r23 = np.linalg.norm(d23, axis=-1)

    v12 = np.linalg.norm(u12, axis=-1)
    v13 = np.linalg.norm(u13, axis=-1)
    v23 = np.linalg.norm(u23, axis=-1)

    # radial speeds: rdot = (d · u) / |d|
    rdot12 = (np.sum(d12 * u12, axis=-1)) / (r12 + eps)
    rdot13 = (np.sum(d13 * u13, axis=-1)) / (r13 + eps)
    rdot23 = (np.sum(d23 * u23, axis=-1)) / (r23 + eps)

    geom = np.stack([r12, r13, r23, v12, v13, v23, rdot12, rdot13, rdot23], axis=-1)
    return geom.astype(np.float32)  # (N,T,9)

def build_features(data: dict, T: int = 200, add_time: bool = True, use_geom: bool = True) -> np.ndarray:
    sl = slice(0, T)
    N = data["aratios_complete"].shape[0]

    # ---- (A) Precomputed series ----
    ar = data["aratios_complete"][:, sl, :].astype(np.float32)         # (N,T,1)
    mr = data["mratios_complete"][:, sl, :].astype(np.float32)         # (N,T,2)
    ms = data["masses_complete"][:, sl, :].astype(np.float32)          # (N,T,3)
    ec = data["eccs_complete"][:, sl, :].astype(np.float32)            # (N,T,2)
    sm = data["smaxes_complete"][:, sl, :].astype(np.float32)          # (N,T,2)
    en = data["energy_complete"][:, sl, :].astype(np.float32)          # (N,T,1)
    am = data["angMomentum_complete"][:, sl, :].astype(np.float32)     # (N,T,3)

    # Angles -> sin/cos  (注意：这里不再使用 tAnos)
    ap  = sincos(data["APs_complete"][:, sl, :])                       # (N,T,4)
    lan = sincos(data["LANs_complete"][:, sl, :])                      # (N,T,4)
    inc = sincos(data["incs_complete"][:, sl, :])                      # (N,T,4)

    # ✅ 删除 periods_complete 和 tAnos_complete
    feats = [ar, mr, ms, ec, sm, en, am, ap, lan, inc]

    # ---- (B) Geometry from raw positions/velocities (invariant) ----
    if use_geom:
        pos = data["positions_complete"][:, sl, :, :].astype(np.float32)   # (N,T,3,3)
        vel = data["velocities_complete"][:, sl, :, :].astype(np.float32)  # (N,T,3,3)
        geom = pairwise_geom(pos, vel)                                     # (N,T,9)
        feats.append(geom)

    # ---- (C) Time channel ----
    if add_time:
        t = np.linspace(0.0, 1.0, T, dtype=np.float32)[None, :, None]      # (1,T,1)
        t = np.repeat(t, N, axis=0)                                        # (N,T,1)
        feats.append(t)

    X = np.concatenate(feats, axis=-1)   # (N,T,C)
    X = X.transpose(0, 2, 1)             # (N,C,T)
    return X.astype(np.float32)
# ---------------------------
# Dataset / Model
# ---------------------------

class SeqDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.astype(np.float32), dtype=torch.float32).view(-1, 1)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

class CNN1D(nn.Module):
    """
    A small, strong baseline 1D CNN for (B, C, T).
    """
    def __init__(self, in_ch: int):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(in_ch, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=5, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),      
        )

    def forward(self, x):
        return self.head(self.backbone(x))

# ---------------------------
# Training / Evaluation
# ---------------------------

def standardize_channels_train_only(X_tr: np.ndarray, X_te: np.ndarray, eps: float = 1e-6):
    """
    X_*: (N,C,T)
    Standardize each channel using mean/std over (N,T) from training split only.
    """
    mu = X_tr.mean(axis=(0, 2), keepdims=True)
    sd = X_tr.std(axis=(0, 2), keepdims=True) + eps
    X_tr = (X_tr - mu) / sd
    X_te = (X_te - mu) / sd
    return X_tr, X_te, (mu, sd)

@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_prob, all_y = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).detach().cpu()
        prob = torch.sigmoid(logits).numpy().ravel()
        all_prob.append(prob)
        all_y.append(yb.numpy().ravel())
    all_prob = np.concatenate(all_prob)
    all_y = np.concatenate(all_y)

    pred = (all_prob >= 0.5).astype(np.int32)
    auc = roc_auc_score(all_y, all_prob)
    f1 = f1_score(all_y, pred)
    acc = accuracy_score(all_y, pred)
    return auc, f1, acc

def train_cnn(
    data: dict,
    y: np.ndarray,
    T: int = 100,
    test_size: float = 0.2,
    seed: int = 42,
    epochs: int = 20,
    batch_size: int = 256,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
):
    """
    Returns: model, (mu, sd) for channel standardization
    """
    X = build_features(data, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y = np.asarray(y).astype(np.float32)
    assert X.shape[0] == y.shape[0], "X and y must have same N"

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )

    X_tr, X_te, norm = standardize_channels_train_only(X_tr, X_te)

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)
    test_loader  = DataLoader(SeqDataset(X_te, y_te), batch_size=batch_size, shuffle=False, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CNN1D(in_ch=X_tr.shape[1]).to(device)

    # Class imbalance handling: pos_weight = #neg/#pos
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            opt.step()

        auc, f1, acc = eval_model(model, test_loader, device)
        print(f"epoch {ep:02d} | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")

    return model, norm


In [9]:
import numpy as np

T = 200

# 1) 先把 df 排到和 data 拼接一致（推荐）
df = df.sort_values(["num_batch", "index"], ascending=True).reset_index(drop=True)

# 2) 计算要保留的样本 mask（长度=原始N，比如40000）
bad_pos = ~np.isfinite(data["positions_complete"][:, :T, :, :]).all(axis=(1,2,3))
bad_vel = ~np.isfinite(data["velocities_complete"][:, :T, :, :]).all(axis=(1,2,3))
mask = ~(bad_pos | bad_vel)

# 3) 同步过滤 data 和 df（被去掉的样本也就没了）
data = {k: v[mask] for k, v in data.items()}
df  = df.loc[mask].reset_index(drop=True)

# 4) 现在再生成 y（自然就只剩保留样本的 y）
y = np.isclose(df["time_unbound"].to_numpy(), 5000.0, rtol=0, atol=1e-6).astype(np.int32)

# 5) 检查
assert len(y) == data["times_complete"].shape[0]
print("After cleaning:", len(y), "pos:", int(y.sum()))

import numpy as np

pos = int((y == 1).sum())
neg = int((y == 0).sum())
print("pos:", pos, "neg:", neg, "total:", len(y))



After cleaning: 39501 pos: 14115
pos: 14115 neg: 25386 total: 39501


In [ ]:
# 可选：再确认一下 X 没 NaN/Inf（尤其你还在用 geom 的话）
Xtmp = build_features(data, T=T, add_time=True, use_geom=True)
print("X:", Xtmp.shape, "finite:", np.isfinite(Xtmp).all())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# 开始训练
model, (mu, sd) = train_cnn(
    data=data,
    y=y,
    T=T,
    epochs=50,
    batch_size=256,
    lr=1e-3,
    use_geom=True,
    add_time=True,
)

X: (39501, 36, 200) finite: True
Using device: cuda
epoch 01 | AUC 0.9288 | F1 0.8100 | Acc 0.8366
epoch 02 | AUC 0.9380 | F1 0.8253 | Acc 0.8544
epoch 03 | AUC 0.9392 | F1 0.8287 | Acc 0.8591
epoch 04 | AUC 0.9423 | F1 0.8325 | Acc 0.8627
epoch 05 | AUC 0.9384 | F1 0.8292 | Acc 0.8589
epoch 06 | AUC 0.9438 | F1 0.8381 | Acc 0.8675
epoch 07 | AUC 0.9492 | F1 0.8421 | Acc 0.8708
epoch 08 | AUC 0.9472 | F1 0.8330 | Acc 0.8617
epoch 09 | AUC 0.9460 | F1 0.8336 | Acc 0.8622
epoch 10 | AUC 0.9496 | F1 0.8456 | Acc 0.8761
epoch 11 | AUC 0.9527 | F1 0.8468 | Acc 0.8766
epoch 12 | AUC 0.9493 | F1 0.8461 | Acc 0.8791
epoch 13 | AUC 0.9508 | F1 0.8431 | Acc 0.8732
epoch 14 | AUC 0.9508 | F1 0.8458 | Acc 0.8755
epoch 15 | AUC 0.9533 | F1 0.8494 | Acc 0.8794
epoch 16 | AUC 0.9556 | F1 0.8475 | Acc 0.8770
epoch 17 | AUC 0.9515 | F1 0.8463 | Acc 0.8774
epoch 18 | AUC 0.9541 | F1 0.8435 | Acc 0.8713
epoch 19 | AUC 0.9535 | F1 0.8486 | Acc 0.8793
epoch 20 | AUC 0.9548 | F1 0.8442 | Acc 0.8727
epoch 21

AttributeError: 'tuple' object has no attribute 'to'

## Attention

In [18]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding for (B, T, D)."""
    def __init__(self, d_model: int, max_len: int = 1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)  # (max_len, D)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len,1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # even
        pe[:, 1::2] = torch.cos(position * div_term)  # odd
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,T,D)
        T = x.size(1)
        return x + self.pe[:, :T, :]

class AttnClassifier(nn.Module):
    """
    Transformer encoder classifier for inputs (B, C, T).
    - token = one time step
    - output = sequence-level binary logit
    """
    def __init__(
        self,
        in_ch: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 3,
        dim_ff: int = 512,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" or "cls"
        max_len: int = 1000,
    ):
        super().__init__()
        assert pooling in ("mean", "cls")

        self.pooling = pooling
        self.in_proj = nn.Linear(in_ch, d_model)

        if pooling == "cls":
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        else:
            self.cls = None

        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # (B,T,D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(0.3),
            nn.Linear(d_model, 1),
        )

        # init
        nn.init.xavier_uniform_(self.in_proj.weight)
        nn.init.zeros_(self.in_proj.bias)
        if self.cls is not None:
            nn.init.normal_(self.cls, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B,C,T) -> (B,T,C)
        x = x.transpose(1, 2)
        x = self.in_proj(x)  # (B,T,D)

        if self.pooling == "cls":
            B = x.size(0)
            cls_tok = self.cls.expand(B, -1, -1)  # (B,1,D)
            x = torch.cat([cls_tok, x], dim=1)    # (B,T+1,D)

        x = self.pos(x)
        x = self.drop(x)
        x = self.encoder(x)  # (B,T(or T+1),D)

        if self.pooling == "cls":
            seq = x[:, 0, :]          # CLS
        else:
            seq = x.mean(dim=1)       # mean pooling over time

        return self.head(seq)         # (B,1)

def train_attn(
    data: dict,
    y: np.ndarray,
    T: int = 100,
    test_size: float = 0.2,
    seed: int = 42,
    epochs: int = 50,
    batch_size: int = 64,
    lr: float = 1e-3,
    add_time: bool = True,
    use_geom: bool = True,
    d_model: int = 128,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 512,
    dropout: float = 0.1,
    pooling: str = "cls",
    use_amp: bool = True,
):
    X = build_features(data, T=T, add_time=add_time, use_geom=use_geom)  # (N,C,T)
    y = np.asarray(y).astype(np.float32)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=seed, stratify=y
    )

    X_tr, X_te, norm = standardize_channels_train_only(X_tr, X_te)

    train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True, drop_last=False)
    test_loader  = DataLoader(SeqDataset(X_te, y_te), batch_size=batch_size, shuffle=False, drop_last=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AttnClassifier(
        in_ch=X_tr.shape[1],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
        max_len=T + (1 if pooling == "cls" else 0) + 5,
    ).to(device)

    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    pos_weight = torch.tensor([neg / (pos + 1e-8)], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits = model(xb)
                loss = loss_fn(logits, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

        auc, f1, acc = eval_model(model, test_loader, device)
        print(f"epoch {ep:02d} | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")

    return model, norm



In [19]:
# 可选：再确认一下 X 没 NaN/Inf（尤其你还在用 geom 的话）
Xtmp = build_features(data, T=T, add_time=True, use_geom=True)
print("X:", Xtmp.shape, "finite:", np.isfinite(Xtmp).all())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# 开始训练
model, (mu, sd) = train_attn(
    data=data,
    y=y,
    T=200,
    epochs=50,
    batch_size=64,   # 8GB 建议 64，不行就 32
    lr=3e-4,
    use_geom=True,
    add_time=True,
    d_model=128,
    nhead=4,
    num_layers=3,
    dim_ff=512,
    dropout=0.1,
    pooling="cls",
    use_amp=True,
)



X: (39501, 36, 200) finite: True
Using device: cuda


/tmp/ipykernel_25250/2577575150.py:61: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
/tmp/ipykernel_25250/2577575150.py:146: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device.type == "cuda"))
/tmp/ipykernel_25250/2577575150.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 01 | AUC 0.9313 | F1 0.8295 | Acc 0.8610


/tmp/ipykernel_25250/2577575150.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 02 | AUC 0.9357 | F1 0.8153 | Acc 0.8400


...

epoch 48 | AUC 0.9598 | F1 0.8522 | Acc 0.8912


/tmp/ipykernel_25250/2577575150.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 49 | AUC 0.9586 | F1 0.8472 | Acc 0.8852


/tmp/ipykernel_25250/2577575150.py:154: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):


epoch 50 | AUC 0.9570 | F1 0.8468 | Acc 0.8851


## SVM

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

T = 100
USE_GEOM = True
ADD_TIME = True
SEED = 42
TEST_SIZE = 0.2

# --- 1) df 排序 + 同步清理（跟你CNN一致：丢掉 pos/vel 非有限的样本） ---
df = df.sort_values(["num_batch", "index"], ascending=True).reset_index(drop=True)

bad_pos = ~np.isfinite(data["positions_complete"][:, :T, :, :]).all(axis=(1,2,3))
bad_vel = ~np.isfinite(data["velocities_complete"][:, :T, :, :]).all(axis=(1,2,3))
mask = ~(bad_pos | bad_vel)

data_clean = {k: v[mask] for k, v in data.items()}
df_clean = df.loc[mask].reset_index(drop=True)

# --- 2) 生成标签 y（与你CNN一致） ---
y = np.isclose(df_clean["time_unbound"].to_numpy(), 5000.0, rtol=0, atol=1e-6).astype(np.int32)

print("After cleaning:", len(y), "pos:", int(y.sum()), "neg:", int(len(y) - y.sum()))

# --- 3) 构造特征：复用你的 build_features -> (N,C,T)，再 flatten -> (N, C*T) ---
X = build_features(data_clean, T=T, add_time=ADD_TIME, use_geom=USE_GEOM)  # (N,C,T)
print("X:", X.shape, "finite:", np.isfinite(X).all())

X2 = X.reshape(X.shape[0], -1).astype(np.float32)  # (N, C*T)

# --- 4) split（和CNN一样 stratify） ---
X_tr, X_te, y_tr, y_te = train_test_split(
    X2, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# --- 5) 线性SVM baseline（高维更合适） ---
# class_weight="balanced" 相当于处理不平衡
svm = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("clf", LinearSVC(C=1.0, class_weight="balanced", max_iter=20000))
])

svm.fit(X_tr, y_tr)

# LinearSVC 没有 predict_proba，用 decision_function 的分数算 AUC
score = svm.decision_function(X_te)           # 连续分数
pred = (score >= 0).astype(np.int32)          # 0 是决策边界

auc = roc_auc_score(y_te, score)
f1 = f1_score(y_te, pred)
acc = accuracy_score(y_te, pred)

print(f"SVM (LinearSVC) | AUC {auc:.4f} | F1 {f1:.4f} | Acc {acc:.4f}")


After cleaning: 39501 pos: 14115 neg: 25386
X: (39501, 36, 100) finite: True
